In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Qwen3-VL-4B-Instruct QLoRA Local Parquet Test Set Evaluation with Error Analysis
"""
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import torch
import pandas as pd
import jiwer
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig
)
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# ===========================
# 1. Core Configurations
# ===========================
CONFIG = {
    'base_model_id': "Qwen/Qwen3-VL-4B-Instruct",
    'lora_path': "/root/autodl-tmp/qwen_lora_output_4b_qlora/final_lora",
    'test_parquet': "/root/autodl-fs/Manchu_OCR/test.parquet",
    'image_dir': "/root/autodl-fs/Manchu_OCR/test",          # Adjust path if necessary
    'prompt_text': "识别满文单词：",
    'min_pixels': 32 * 32,
    'max_pixels': 384 * 384,
    'output_error_csv': "error_analysis.csv",                # Error analysis CSV output path
}

def main():
    print("\n" + "="*80)
    print("Starting Qwen3-VL-4B QLoRA Local Test Set Evaluation & Error Analysis")
    print("="*80)

    # --- 1. Load Data ---
    print(f"\n[1/4] Loading test set: {CONFIG['test_parquet']}")
    df = pd.read_parquet(CONFIG['test_parquet'])
    print(f"Data loaded successfully. Total test samples: {len(df)}")

    # --- 2. Load Model ---
    print("\n[2/4] Loading 4-bit model and LoRA weights...")
    processor = AutoProcessor.from_pretrained(
        CONFIG['lora_path'],
        min_pixels=CONFIG['min_pixels'],
        max_pixels=CONFIG['max_pixels']
    )

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG['base_model_id'],
        torch_dtype=torch.bfloat16,
        device_map="auto",
        quantization_config=quantization_config,
    )

    model = PeftModel.from_pretrained(base_model, CONFIG['lora_path'])
    model.eval()

    # --- 3. Inference ---
    print("\n[3/4] Starting inference...")
    predictions = []
    ground_truths = []
    filenames = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
        img_path = os.path.join(CONFIG['image_dir'], row['filename'])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Warning: Skipping unreadable image: {img_path} | Error: {e}")
            continue

        target_text = row['roman']
        filenames.append(row['filename'])

        # Build inputs
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": CONFIG['prompt_text']},
                ],
            }
        ]

        text_prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        # Generate
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        pred_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        predictions.append(pred_text)
        ground_truths.append(target_text)

    if len(predictions) == 0:
        print("\nError: Evaluation failed. No samples were successfully processed. Please check the image paths.")
        return

    # --- 4. Calculate Overall Metrics ---
    print("\n[4/4] Calculating overall CER/WER...")
    cer = jiwer.cer(ground_truths, predictions)
    wer = jiwer.wer(ground_truths, predictions)

    print("\n" + "="*70)
    print("Overall Evaluation Results")
    print("="*70)
    print(f"Valid samples             : {len(predictions)}")
    print(f"Character Error Rate (CER): {cer * 100:.4f}%")
    print(f"Word Error Rate (WER)     : {wer * 100:.4f}%")
    print("="*70)

    # ========================
    # Complete Error Analysis
    # ========================
    print("\nGenerating detailed error analysis...")

    # Calculate individual CER for each sample
    individual_cers = [jiwer.cer([gt], [pred]) for gt, pred in zip(ground_truths, predictions)]

    # Build error analysis DataFrame
    error_df = pd.DataFrame({
        'filename': filenames,
        'ground_truth': ground_truths,
        'prediction': predictions,
        'cer': individual_cers
    })

    # Save complete error analysis to CSV
    error_df.to_csv(CONFIG['output_error_csv'], index=False, encoding='utf-8')
    print(f"Error analysis CSV saved to -> {CONFIG['output_error_csv']}")

    # Statistics
    print("\n" + "="*70)
    print("Error Analysis Statistical Summary")
    print("="*70)
    print(f"Mean CER                 : {error_df['cer'].mean()*100:.4f}%")
    print(f"Median CER               : {error_df['cer'].median()*100:.4f}%")
    print(f"Standard Deviation       : {error_df['cer'].std()*100:.4f}%")
    print(f"Worst Sample CER         : {error_df['cer'].max()*100:.4f}%")
    print(f"Best Sample CER          : {error_df['cer'].min()*100:.4f}%")
    print(f"Completely Correct Samples: {(error_df['cer'] == 0).sum()} / {len(error_df)}")
    print("="*70)

    # Top 10 worst error cases
    worst_errors = error_df.nlargest(10, 'cer').copy()
    print("\nTop 10 Worst Error Cases (Sorted by individual CER descending):")
    print("-" * 80)
    for i, row in worst_errors.iterrows():
        print(f"[{i+1:2d}] CER = {row['cer']*100:6.2f}%")
        print(f"   Ground Truth : {row['ground_truth']}")
        print(f"   Prediction   : {row['prediction']}")
        print(f"   Filename     : {row['filename']}")
        print("-" * 80)

    # Top 5 error sampling
    print("\nTop 5 Error Samples Quick Check:")
    error_count = 0
    for gt, pred in zip(ground_truths, predictions):
        if gt != pred:
            print(f"   GT  : {gt}")
            print(f"   Pred: {pred}")
            print("   " + "-"*60)
            error_count += 1
            if error_count >= 5:
                break
    if error_count == 0:
        print("No errors found in the sampled batch.")

    print("\nEvaluation and error analysis completed.")
    print(f"Complete error report saved to -> {CONFIG['output_error_csv']}")

if __name__ == "__main__":
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS



Starting Qwen3-VL-4B QLoRA Local Test Set Evaluation & Error Analysis

[1/4] Loading test set: /root/autodl-fs/Manchu_OCR/test.parquet
Data loaded successfully. Total test samples: 218

[2/4] Loading 4-bit model and LoRA weights...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)



[3/4] Starting inference...


Evaluating: 100%|██████████| 218/218 [01:35<00:00,  2.29it/s]


[4/4] Calculating overall CER/WER...

Overall Evaluation Results
Valid samples             : 218
Character Error Rate (CER): 3.3727%
Word Error Rate (WER)     : 13.3028%

Generating detailed error analysis...
Error analysis CSV saved to -> error_analysis.csv

Error Analysis Statistical Summary
Mean CER                 : 3.9581%
Median CER               : 0.0000%
Standard Deviation       : 11.8724%
Worst Sample CER         : 100.0000%
Best Sample CER          : 0.0000%
Completely Correct Samples: 189 / 218

Top 10 Worst Error Cases (Sorted by individual CER descending):
--------------------------------------------------------------------------------
[12] CER = 100.00%
   Ground Truth : da
   Prediction   : pe
   Filename     : 00012.jpg
--------------------------------------------------------------------------------
[112] CER =  50.00%
   Ground Truth : dolo
   Prediction   : dulu
   Filename     : 00112.jpg
------------------------------------------------------------------------------